# 03 Merged Strict+Cloud Feedback GA Search v2 Final

JOILang GA feedback experiment notebook.

## Cell 목차

0. Server preset, path, runtime, helper functions  
1. Local worker and OpenAI preflight  
2. Generate strict+cloud merged feedback with eval pipeline  
3. Inspect `advisor_rich_feedback.json`  
4. Baseline row 1 merged-context GA/advisor run  
5. Feedback / mutation / advisor evidence inspection  
6. Actual prompt and advisor prompt inspection  
7. Manual merged-feedback patch  
8. Before/after diff  
9. Re-test same row  
10. Before/after evaluation and prompt comparison  
11. Category sweep  
12. Full eval pipeline + full 280 rows × 10 generations  
13. Final plots  

## Scope note

`run_eval_pipeline_check.sh full`은 cloudless DET only가 아니라 **local strict DET + cloud semantic judge + strict/cloud merge + feedback schema check**이다.  
이 notebook은 먼저 merged feedback artifact를 만들고, 그 evidence를 보존한 상태에서 GA/advisor 결과를 비교한다.

In [ ]:
# ============================================================
# Cell 0. Server preset, path, runtime, and helper functions
# ============================================================

import os
import sys
import json
import csv
import shlex
import time
import difflib
import subprocess
from pathlib import Path
from datetime import datetime

import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 240)
pd.set_option("display.width", 260)
pd.set_option("display.max_colwidth", 260)

# ------------------------------------------------------------
# 0.1 Server preset
# ------------------------------------------------------------
# A100 default:
#   repo        = /root/llm/JOILang-Server
#   local model = /root/llm/local_models/qwen25_coder_14b
#
# A6000 default:
#   repo        = /home/mgjeong/Desktop/llm/JOILang-Server
#   local model = /home/mgjeong/Desktop/llm/local_models/qwen25_coder_14b
#
# Important:
#   The model path must NOT become:
#   /root/llm/JOILang-Server/local_models/qwen25_coder_14b
SERVER_PRESET = os.environ.get("SERVER_PRESET", "a100")  # "a100" or "a6000"
MODEL_KEY_VALUE = os.environ.get("MODEL_KEY", "qwen25_coder_14b")

if SERVER_PRESET == "a100":
    os.environ["JOILANG_BASE_DIR"] = os.environ.get("JOILANG_BASE_DIR", "/root/llm/JOILang-Server")
    os.environ["JOI_V15_LOCAL_MODEL_BASE_DIR"] = os.environ.get("JOI_V15_LOCAL_MODEL_BASE_DIR", "/root/llm/local_models")
    os.environ["JOI_V15_LOCAL_MODEL_NAME"] = os.environ.get(
        "JOI_V15_LOCAL_MODEL_NAME",
        f"/root/llm/local_models/{MODEL_KEY_VALUE}",
    )
    os.environ["JOI_V15_LOCAL_LOAD_IN_4BIT"] = os.environ.get("JOI_V15_LOCAL_LOAD_IN_4BIT", "false")

elif SERVER_PRESET == "a6000":
    os.environ["JOILANG_BASE_DIR"] = os.environ.get("JOILANG_BASE_DIR", "/home/mgjeong/Desktop/llm/JOILang-Server")
    os.environ["JOI_V15_LOCAL_MODEL_BASE_DIR"] = os.environ.get(
        "JOI_V15_LOCAL_MODEL_BASE_DIR",
        "/home/mgjeong/Desktop/llm/local_models",
    )
    os.environ["JOI_V15_LOCAL_MODEL_NAME"] = os.environ.get(
        "JOI_V15_LOCAL_MODEL_NAME",
        f"/home/mgjeong/Desktop/llm/local_models/{MODEL_KEY_VALUE}",
    )
    os.environ["JOI_V15_LOCAL_LOAD_IN_4BIT"] = os.environ.get("JOI_V15_LOCAL_LOAD_IN_4BIT", "true")

else:
    raise ValueError(f"Unknown SERVER_PRESET={SERVER_PRESET!r}")

os.environ["MODEL_KEY"] = MODEL_KEY_VALUE
os.environ["JOI_V15_LOCAL_DEVICE"] = os.environ.get("JOI_V15_LOCAL_DEVICE", "cuda:0")
os.environ["JOI_V15_LOCAL_FILES_ONLY"] = os.environ.get("JOI_V15_LOCAL_FILES_ONLY", "true")
os.environ["JOI_V15_LOCAL_DTYPE"] = os.environ.get("JOI_V15_LOCAL_DTYPE", "bf16")
os.environ["JOI_V15_LOCAL_TRUST_REMOTE_CODE"] = os.environ.get("JOI_V15_LOCAL_TRUST_REMOTE_CODE", "true")

# ------------------------------------------------------------
# 0.2 Core paths
# ------------------------------------------------------------
BASE_DIR = Path(os.environ["JOILANG_BASE_DIR"]).expanduser().resolve()
VERSION_ROOT = BASE_DIR / "gpt_mg" / "version0_15_update20260413"

GA_SCRIPT = VERSION_ROOT / "scripts" / "run_ga_search.py"
RUN_BENCHMARK = VERSION_ROOT / "scripts" / "run_benchmark.py"
EVAL_PIPELINE = BASE_DIR / "run_eval_pipeline_check.sh"

DATASET = BASE_DIR / "datasets" / "JOICommands-280.csv"
SERVICE_SCHEMA = BASE_DIR / "datasets" / "service_list_ver2.0.1.json"
DEFAULT_GENOME = VERSION_ROOT / "genomes" / "example_genome.json"

PYTHON = os.environ.get("JOI_V15_PYTHON", sys.executable)
WORKER_PYTHON = os.environ.get("JOI_V15_WORKER_PYTHON", PYTHON)

MODEL_KEY = os.environ["MODEL_KEY"]
DEVICE = os.environ["JOI_V15_LOCAL_DEVICE"]

LOCAL_MODELS_BASE = Path(os.environ["JOI_V15_LOCAL_MODEL_BASE_DIR"]).expanduser().resolve()
LOCAL_MODEL_DIR = Path(os.environ["JOI_V15_LOCAL_MODEL_NAME"]).expanduser().resolve()

RUN_TAG = os.environ.get("RUN_TAG", datetime.now().strftime("%Y%m%d_%H%M%S"))
NOTEBOOK_RUN_ROOT = BASE_DIR / "artifacts" / "notebook_ga_runs" / RUN_TAG
NOTEBOOK_RUN_ROOT.mkdir(parents=True, exist_ok=True)

LLM_EXTRA_JSON = NOTEBOOK_RUN_ROOT / "llm_extra_worker_preflight.json"
LLM_EXTRA_JSON.write_text(json.dumps({
    "local_model_name": str(LOCAL_MODEL_DIR),
    "local_files_only": str(os.environ.get("JOI_V15_LOCAL_FILES_ONLY", "true")).lower() == "true",
    "local_device": DEVICE,
    "local_dtype": os.environ.get("JOI_V15_LOCAL_DTYPE", "bf16"),
    "local_load_in_4bit": str(os.environ.get("JOI_V15_LOCAL_LOAD_IN_4BIT", "false")).lower() == "true",
    "local_trust_remote_code": str(os.environ.get("JOI_V15_LOCAL_TRUST_REMOTE_CODE", "true")).lower() == "true",
}, ensure_ascii=False, indent=2), encoding="utf-8")

# ------------------------------------------------------------
# 0.3 Environment passed to subprocesses
# ------------------------------------------------------------
ENV = os.environ.copy()
ENV.update({
    "JOI_V15_PYTHON": PYTHON,
    "JOI_V15_WORKER_PYTHON": WORKER_PYTHON,

    "JOI_V15_LOCAL_MODEL_BASE_DIR": str(LOCAL_MODELS_BASE),
    "JOI_V15_LOCAL_MODEL_NAME": str(LOCAL_MODEL_DIR),
    "JOI_V15_LOCAL_FILES_ONLY": os.environ.get("JOI_V15_LOCAL_FILES_ONLY", "true"),
    "JOI_V15_LOCAL_DEVICE": DEVICE,
    "JOI_V15_LOCAL_DTYPE": os.environ.get("JOI_V15_LOCAL_DTYPE", "bf16"),
    "JOI_V15_LOCAL_LOAD_IN_4BIT": os.environ.get("JOI_V15_LOCAL_LOAD_IN_4BIT", "false"),
    "JOI_V15_LOCAL_TRUST_REMOTE_CODE": os.environ.get("JOI_V15_LOCAL_TRUST_REMOTE_CODE", "true"),

    "TRANSFORMERS_VERBOSITY": "error",
    "HF_HUB_DISABLE_PROGRESS_BARS": "1",
    "TOKENIZERS_PARALLELISM": "false",
    "PYTHONFAULTHANDLER": "1",
})

# Cloud API env aliases
OPENAI_KEY = (
    os.environ.get("OPENAI_API_KEY_PROJ_BENCH")
    or os.environ.get("JOI_EVAL_OPENAI_API_KEY")
    or os.environ.get("JOI_V15_OPENAI_API_KEY")
    or os.environ.get("OPENAI_API_KEY")
)
if OPENAI_KEY:
    ENV["OPENAI_API_KEY"] = OPENAI_KEY
    ENV["OPENAI_API_KEY_PROJ_BENCH"] = OPENAI_KEY
    ENV["JOI_EVAL_OPENAI_API_KEY"] = OPENAI_KEY
    ENV["JOI_V15_OPENAI_API_KEY"] = OPENAI_KEY
ENV["LANGSMITH_TRACING"] = ENV.get("LANGSMITH_TRACING", "false")
ENV["LANGCHAIN_TRACING_V2"] = ENV.get("LANGCHAIN_TRACING_V2", "false")

# ------------------------------------------------------------
# 0.4 Sanity print and assertions
# ------------------------------------------------------------
print("SERVER_PRESET:", SERVER_PRESET)
print("BASE_DIR:", BASE_DIR, BASE_DIR.exists())
print("VERSION_ROOT:", VERSION_ROOT, VERSION_ROOT.exists())
print("GA_SCRIPT:", GA_SCRIPT, GA_SCRIPT.exists())
print("RUN_BENCHMARK:", RUN_BENCHMARK, RUN_BENCHMARK.exists())
print("EVAL_PIPELINE:", EVAL_PIPELINE, EVAL_PIPELINE.exists())
print("DATASET:", DATASET, DATASET.exists())
print("SERVICE_SCHEMA:", SERVICE_SCHEMA, SERVICE_SCHEMA.exists())
print("DEFAULT_GENOME:", DEFAULT_GENOME, DEFAULT_GENOME.exists())
print("MODEL_KEY:", MODEL_KEY)
print("LOCAL_MODELS_BASE:", LOCAL_MODELS_BASE, LOCAL_MODELS_BASE.exists())
print("LOCAL_MODEL_DIR:", LOCAL_MODEL_DIR, LOCAL_MODEL_DIR.exists())
print("DEVICE:", DEVICE)
print("JOI_V15_LOCAL_LOAD_IN_4BIT:", ENV["JOI_V15_LOCAL_LOAD_IN_4BIT"])
print("LLM_EXTRA_JSON:", LLM_EXTRA_JSON)
print("NOTEBOOK_RUN_ROOT:", NOTEBOOK_RUN_ROOT)
print("OpenAI key configured:", bool(OPENAI_KEY))

assert BASE_DIR.exists(), BASE_DIR
assert VERSION_ROOT.exists(), VERSION_ROOT
assert GA_SCRIPT.exists(), GA_SCRIPT
assert RUN_BENCHMARK.exists(), RUN_BENCHMARK
assert DATASET.exists(), DATASET
assert SERVICE_SCHEMA.exists(), SERVICE_SCHEMA
assert DEFAULT_GENOME.exists(), DEFAULT_GENOME

if not LOCAL_MODEL_DIR.exists():
    print("\n[ERROR] LOCAL_MODEL_DIR does not exist.")
    print("Expected:", LOCAL_MODEL_DIR)
    print("\nAvailable dirs under LOCAL_MODELS_BASE:")
    if LOCAL_MODELS_BASE.exists():
        for p in sorted(LOCAL_MODELS_BASE.iterdir()):
            print(" -", p)
    raise FileNotFoundError(LOCAL_MODEL_DIR)

assert str(LOCAL_MODEL_DIR) != "/root/llm/JOILang-Server/local_models/qwen25_coder_14b", (
    "Wrong A100 model path: it points inside repo/local_models."
)

# ------------------------------------------------------------
# 0.5 Helpers
# ------------------------------------------------------------
def ts():
    return datetime.now().strftime("%Y%m%d_%H%M%S")

def run_cmd(cmd, *, cwd=BASE_DIR, env=ENV, log_path=None, check=True):
    cmd = [str(x) for x in cmd]
    print("\n[CMD]")
    print(" ".join(shlex.quote(x) for x in cmd))

    if log_path is not None:
        log_path = Path(log_path)
        log_path.parent.mkdir(parents=True, exist_ok=True)
        print("[LOG]", log_path)

    proc = subprocess.Popen(
        cmd,
        cwd=str(cwd),
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    lines = []
    with (open(log_path, "w", encoding="utf-8") if log_path else open(os.devnull, "w", encoding="utf-8")) as lf:
        assert proc.stdout is not None
        for line in proc.stdout:
            print(line, end="")
            lines.append(line)
            if log_path:
                lf.write(line)

    rc = proc.wait()
    if check and rc != 0:
        raise RuntimeError(f"command failed rc={rc}: {' '.join(cmd)}")
    return rc, "".join(lines)

def load_json(path):
    path = Path(path)
    if not path.exists():
        return {}
    return json.loads(path.read_text(encoding="utf-8"))

def dump_json(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding="utf-8")
    return path

def read_csv_if_exists(path):
    path = Path(path)
    if not path.exists():
        return pd.DataFrame()
    return pd.read_csv(path)

def latest_file(root, pattern):
    files = sorted(Path(root).glob(pattern), key=lambda p: p.stat().st_mtime)
    return files[-1] if files else None

def ga_common_args(genome_json=None, mutation_mode="cloudless_decompiler"):
    genome_json = Path(genome_json or DEFAULT_GENOME).expanduser().resolve()
    return [
        PYTHON, "-u", str(GA_SCRIPT),
        "--profile", "version0_15",
        "--genome-json", str(genome_json),
        "--dataset", str(DATASET),
        "--service-schema", str(SERVICE_SCHEMA),
        "--model-key", MODEL_KEY,
        "--llm-mode", "worker",
        "--candidate-k", "1",
        "--repair-attempts", "0",
        "--det-profile", "strict",

        "--selection-mode", "redesign",
        "--fitness-mode", "phase_aware",
        "--mutation-mode", mutation_mode,
        "--category-balance-mode", "guard",
        "--token-penalty-mode", "hybrid",
        "--stop-controller-mode", "active",
        "--reasoning-mutation-mode", "auto",
        "--intent-hint-mode", "auto",

        "--feedback-guided-mutation",
        "--enable-compression-mutation",
        "--enable-prompt-decompiler",
        "--enable-rendered-prompt-dedupe",
        "--enable-pareto-archive",
        "--enable-group-specialist-archives",

        "--full-run",
        "--force",
        "--progress", "verbose",
        "--retries", "0",
        "--target-detpass", "90",
    ]

def run_ga(
    label,
    scope_args,
    tuning_args=None,
    extra_args=None,
    output_root=None,
    genome_json=None,
    mutation_mode="cloudless_decompiler",
    check=True,
):
    output_root = Path(output_root or (NOTEBOOK_RUN_ROOT / label)).resolve()
    output_root.mkdir(parents=True, exist_ok=True)
    log_path = output_root / f"{label}.log"

    cmd = ga_common_args(genome_json=genome_json, mutation_mode=mutation_mode)
    cmd += list(scope_args)
    cmd += list(tuning_args or [])
    cmd += ["--output-root", str(output_root)]
    cmd += list(extra_args or [])

    rc, output = run_cmd(cmd, log_path=log_path, check=check)
    return output_root

def run_worker_preflight(check=False):
    log_path = NOTEBOOK_RUN_ROOT / f"worker_preflight_{ts()}.log"
    cmd = [
        PYTHON, str(RUN_BENCHMARK),
        "--suite", "paper_local5",
        "--model-key", MODEL_KEY,
        "--prompt-render-mode", "blocks",
        "--llm-extra-json", str(LLM_EXTRA_JSON),
        "--preflight-only",
        "--print-worker-info",
        "--strict-availability",
    ]
    return run_cmd(cmd, log_path=log_path, check=check)

def collect_candidate_tables(run_dir):
    run_dir = Path(run_dir)
    search_roots = [run_dir / "candidates", run_dir]
    skip_names = {
        "advisor_mutation_summary.csv",
        "ga_generation_progress.csv",
        "category_summary.csv",
        "failure_reason_summary.csv",
        "suite_summary.csv",
        "main_model_comparison.csv",
        "tradeoff_summary.csv",
        "pareto_rows.csv",
        "pareto_summary.csv",
    }
    dfs = []
    seen = set()
    for root in search_roots:
        if not root.exists():
            continue
        for p in sorted(root.glob("*.csv")):
            if p.name in skip_names or p in seen:
                continue
            seen.add(p)
            try:
                df = pd.read_csv(p)
                df["source_file"] = str(p)
                dfs.append(df)
            except Exception as e:
                print("failed:", p, e)
    return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

def summarize_ga_run(run_dir):
    run_dir = Path(run_dir)
    summary = load_json(run_dir / "ga_summary.json")
    best = load_json(run_dir / "best_genome.json")
    print("RUN_DIR:", run_dir)
    print("best_DETPass:", summary.get("best_DETPass") or summary.get("accepted_best_DETPass"))
    print("best_avg_DET:", summary.get("best_avg_DET") or summary.get("accepted_best_avg_DET"))
    print("best_prompt_tokens:", summary.get("best_avg_prompt_tokens") or summary.get("accepted_best_avg_prompt_tokens"))
    print("best_genome_id:", best.get("id") or best.get("genome_id"))
    print("stop_reason:", summary.get("stop_reason"))
    for name in [
        "ga_summary.json",
        "best_genome.json",
        "ga_block_diffs.jsonl",
        "advisor_mutation_summary.csv",
        "ga_generation_progress.csv",
        "pareto_rows.csv",
    ]:
        p = run_dir / name
        print(f"{name}:", p.exists(), p)
    return summary, best

def collect_run_table(run_dirs):
    rows = []
    for rd in map(Path, run_dirs):
        s = load_json(rd / "ga_summary.json")
        b = load_json(rd / "best_genome.json")
        rows.append({
            "run_dir": str(rd),
            "label": rd.name,
            "best_DETPass": s.get("best_DETPass") or s.get("accepted_best_DETPass"),
            "best_avg_DET": s.get("best_avg_DET") or s.get("accepted_best_avg_DET"),
            "best_prompt_tokens": s.get("best_avg_prompt_tokens") or s.get("accepted_best_avg_prompt_tokens"),
            "stop_reason": s.get("stop_reason"),
            "best_genome_id": b.get("id") or b.get("genome_id"),
        })
    return pd.DataFrame(rows)

def generation_history(run_dir):
    run_dir = Path(run_dir)
    progress_csv = run_dir / "ga_generation_progress.csv"
    if progress_csv.exists():
        df = pd.read_csv(progress_csv)
        df["run_dir"] = str(run_dir)
        return df
    s = load_json(run_dir / "ga_summary.json")
    hist = s.get("best_history") or s.get("generation_history") or []
    if not hist:
        return pd.DataFrame()
    df = pd.DataFrame(hist)
    df["run_dir"] = str(run_dir)
    return df

def plot_generation_history(run_dirs):
    hdfs = [generation_history(rd) for rd in run_dirs]
    hdfs = [df for df in hdfs if not df.empty]
    if not hdfs:
        print("No generation history found.")
        return pd.DataFrame()

    hist = pd.concat(hdfs, ignore_index=True)
    display(hist.head())

    gen_col = "generation" if "generation" in hist.columns else hist.columns[0]

    det_col = next((c for c in [
        "validation_det_pass_rate", "train_det_pass_rate", "best_DETPass", "DETPass"
    ] if c in hist.columns), None)

    avg_col = next((c for c in [
        "validation_avg_det_score", "avg_det_score", "best_avg_DET", "avg_DET"
    ] if c in hist.columns), None)

    tok_col = next((c for c in [
        "avg_prompt_tokens", "best_avg_prompt_tokens", "prompt_tokens"
    ] if c in hist.columns), None)

    if det_col:
        plt.figure(figsize=(8, 4))
        for rd, g in hist.groupby("run_dir"):
            plt.plot(g[gen_col], g[det_col], marker="o", label=Path(rd).name)
        plt.xlabel("Generation")
        plt.ylabel(det_col)
        plt.title("GA DETPass by generation")
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.show()

    if avg_col:
        plt.figure(figsize=(8, 4))
        for rd, g in hist.groupby("run_dir"):
            plt.plot(g[gen_col], g[avg_col], marker="o", label=Path(rd).name)
        plt.xlabel("Generation")
        plt.ylabel(avg_col)
        plt.title("GA average DET by generation")
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.show()

    if tok_col:
        plt.figure(figsize=(8, 4))
        for rd, g in hist.groupby("run_dir"):
            plt.plot(g[gen_col], g[tok_col], marker="o", label=Path(rd).name)
        plt.xlabel("Generation")
        plt.ylabel(tok_col)
        plt.title("Prompt-token trend by generation")
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.show()

    return hist

def inspect_failures(run_dir, max_rows=80):
    df = collect_candidate_tables(run_dir)
    if df.empty:
        print("No candidate CSV rows found.")
        return df

    cols = [
        c for c in df.columns
        if any(k in c.lower() for k in [
            "row", "category", "det", "pass", "failure",
            "error", "prompt", "token", "candidate", "genome"
        ])
    ]
    display(df[cols].head(max_rows))

    if "generation_error_type" in df.columns:
        print("\nGeneration error counts:")
        display(df["generation_error_type"].fillna("").value_counts().to_frame("count"))

    return df

def detect_worker_crash(run_dir):
    df = collect_candidate_tables(run_dir)
    if df.empty:
        return False
    text = df.astype(str).to_string()
    return "worker_crash" in text or "Repo id must be in the form" in text

def compare_two_runs(run_a, run_b):
    a = collect_candidate_tables(run_a)
    b = collect_candidate_tables(run_b)
    if a.empty or b.empty:
        print("candidate table missing")
        return pd.DataFrame()

    key_candidates = ["row_no", "row_id", "dataset_row", "index"]
    key = next((k for k in key_candidates if k in a.columns and k in b.columns), None)
    if key is None:
        print("No common row key. Showing run-level summaries only.")
        display(collect_run_table([run_a, run_b]))
        return pd.DataFrame()

    def pick_cols(df):
        return [key] + [
            c for c in df.columns
            if c != key and any(k in c.lower() for k in ["pass", "det", "failure", "error", "candidate", "token"])
        ]

    merged = a[pick_cols(a)].merge(b[pick_cols(b)], on=key, suffixes=("_before", "_after"))
    display(merged.head(100))
    return merged

def inspect_prompt_log_from_candidates(run_dir, row_index=0, max_chars=8000):
    df = collect_candidate_tables(run_dir)
    if df.empty:
        print("No candidate table.")
        return None

    if "prompt_log_paths" not in df.columns:
        print("No prompt_log_paths column.")
        display(df.head())
        return None

    raw = df.iloc[row_index]["prompt_log_paths"]
    try:
        paths = json.loads(raw)
    except Exception:
        print("Could not parse prompt_log_paths:", raw)
        return None

    if not paths:
        print("empty prompt_log_paths")
        return None

    p = Path(paths[0])
    print("prompt log:", p, p.exists())
    if not p.exists():
        return None

    log = json.loads(p.read_text(encoding="utf-8"))
    request = log.get("request", {})
    system = request.get("system", "")
    user = request.get("user", "")

    print("\n=== SYSTEM HEAD ===")
    print(system[:max_chars])
    print("\n=== USER HEAD ===")
    print(user[:max_chars])
    print("\n=== USER TAIL ===")
    print(user[-max_chars:])

    return log

def show_prompt_mutation_evidence(run_dir):
    run_dir = Path(run_dir)
    print("=== ga_summary ===")
    summary = load_json(run_dir / "ga_summary.json")
    print(json.dumps({
        "best_DETPass": summary.get("best_DETPass") or summary.get("accepted_best_DETPass"),
        "best_avg_DET": summary.get("best_avg_DET") or summary.get("accepted_best_avg_DET"),
        "best_prompt_tokens": summary.get("best_avg_prompt_tokens") or summary.get("accepted_best_avg_prompt_tokens"),
        "stop_reason": summary.get("stop_reason"),
    }, ensure_ascii=False, indent=2))

    print("\n=== best_genome ===")
    best = load_json(run_dir / "best_genome.json")
    print("id:", best.get("id") or best.get("genome_id"))
    print("blocks:", best.get("blocks"))
    print("params:")
    print(json.dumps(best.get("params", {}), ensure_ascii=False, indent=2)[:4000])
    print("block_params:")
    print(json.dumps(best.get("block_params", {}), ensure_ascii=False, indent=2)[:8000])

    print("\n=== ga_block_diffs.jsonl ===")
    diff_path = run_dir / "ga_block_diffs.jsonl"
    if diff_path.exists():
        rows = [json.loads(x) for x in diff_path.read_text(encoding="utf-8").splitlines() if x.strip()]
        df = pd.DataFrame(rows)
        display(df.head(200))
    else:
        print("missing:", diff_path)

    print("\n=== referenced generated block files ===")
    for bid, params in (best.get("block_params") or {}).items():
        sf = params.get("source_file")
        if not sf:
            continue
        p = VERSION_ROOT / "blocks" / sf
        print("\nBLOCK", bid)
        print("source_file:", p, p.exists())
        if p.exists():
            txt = p.read_text(encoding="utf-8")
            for marker in [
                "AUTO-PATCH MICRO-RULES",
                "MANUAL FOCUS RULES",
                "AUTO-GENERATED EXEMPLARS FROM GT FAILURES",
                "NOTEBOOK MANUAL FOCUS RULES",
            ]:
                if marker in txt:
                    print("contains:", marker)
            print("\n--- tail of block text ---")
            print(txt[-3000:])

def resolve_block_source_file(genome, block_id):
    params = (genome.get("block_params") or {}).get(block_id, {}) or {}
    source_file = params.get("source_file")
    if source_file:
        p = VERSION_ROOT / "blocks" / source_file
        if p.exists():
            return p

    candidates = sorted((VERSION_ROOT / "blocks").glob(f"{block_id}_*.txt"))
    if candidates:
        return candidates[0]

    candidates = sorted((VERSION_ROOT / "blocks").glob(f"{block_id}*.txt"))
    if candidates:
        return candidates[0]

    raise FileNotFoundError(f"Cannot resolve block source for block_id={block_id}")

def make_manual_patched_genome(
    base_genome_path,
    *,
    rules,
    target_block_id="02",
    patch_name=None,
    force_source_file_patch=True,
):
    base_genome_path = Path(base_genome_path)
    genome = load_json(base_genome_path)
    if not genome:
        raise ValueError(f"empty genome: {base_genome_path}")

    patch_name = patch_name or f"manual_patch_{target_block_id}_{ts()}"
    old_id = genome.get("id", "genome")
    genome["id"] = f"{old_id}__{patch_name}"
    genome.setdefault("block_params", {})
    genome["block_params"].setdefault(target_block_id, {})

    # 1) Add micro_rules to genome metadata.
    old_rules = list(genome["block_params"][target_block_id].get("micro_rules") or [])
    merged_rules = old_rules[:]
    for rule in rules:
        if rule not in merged_rules:
            merged_rules.append(rule)
    genome["block_params"][target_block_id]["micro_rules"] = merged_rules[-12:]

    # 2) Force source_file patch so the rules are definitely visible in final rendered prompt.
    if force_source_file_patch:
        source_path = resolve_block_source_file(genome, target_block_id)
        source_text = source_path.read_text(encoding="utf-8")
        rule_text = "\n".join(f"- {rule}" for rule in rules)
        marker = "NOTEBOOK MANUAL FOCUS RULES"
        patched_text = source_text.rstrip() + f"\n\n{marker}\n{rule_text}\n"

        rel = f"generated/{source_path.stem}__{patch_name}{source_path.suffix or '.txt'}"
        out_block = VERSION_ROOT / "blocks" / rel
        out_block.parent.mkdir(parents=True, exist_ok=True)
        out_block.write_text(patched_text, encoding="utf-8")

        genome["block_params"][target_block_id]["source_file"] = rel
        genome["_notebook_manual_block_patch"] = {
            "target_block_id": target_block_id,
            "source_path": str(source_path),
            "patched_block_path": str(out_block),
            "marker": marker,
        }

    genome["_notebook_manual_patch"] = {
        "created_at": datetime.now().isoformat(timespec="seconds"),
        "target_block_id": target_block_id,
        "rules": rules,
        "base_genome_path": str(base_genome_path),
        "force_source_file_patch": force_source_file_patch,
    }

    out_path = NOTEBOOK_RUN_ROOT / "patched_genomes" / f"{patch_name}.json"
    dump_json(out_path, genome)
    return out_path

def diff_json_objects(before_path, after_path, max_lines=220):
    before_path = Path(before_path)
    after_path = Path(after_path)
    before = json.dumps(load_json(before_path), ensure_ascii=False, indent=2, sort_keys=True).splitlines()
    after = json.dumps(load_json(after_path), ensure_ascii=False, indent=2, sort_keys=True).splitlines()

    diff = list(difflib.unified_diff(
        before,
        after,
        fromfile=str(before_path),
        tofile=str(after_path),
        lineterm="",
    ))
    print("\n".join(diff[:max_lines]))
    if len(diff) > max_lines:
        print(f"\n... truncated: {len(diff) - max_lines} more lines")
    return diff

def advisor_effectiveness_report(run_dir):
    run_dir = Path(run_dir)
    summary = load_json(run_dir / "ga_summary.json")
    advisor_csv = read_csv_if_exists(run_dir / "advisor_mutation_summary.csv")
    diffs = run_dir / "ga_block_diffs.jsonl"

    report = {
        "advisor_used": summary.get("advisor_used"),
        "advisor_proposals_generated": summary.get("advisor_proposals_generated"),
        "advisor_proposals_accepted_applied": summary.get("advisor_proposals_accepted_applied"),
        "advisor_children_scheduled": summary.get("advisor_children_scheduled"),
        "advisor_compression_children_scheduled": summary.get("advisor_compression_children_scheduled"),
        "advisor_csv_rows": len(advisor_csv),
        "accepted_rows": 0,
        "no_advisor_proposals_parsed_rows": 0,
        "advisor_backed_diff_count": 0,
    }

    if not advisor_csv.empty:
        if "accepted" in advisor_csv:
            report["accepted_rows"] = int(advisor_csv["accepted"].astype(str).str.lower().isin(["true", "1", "yes"]).sum())
        if "rejection_reason" in advisor_csv:
            report["no_advisor_proposals_parsed_rows"] = int((advisor_csv["rejection_reason"].astype(str) == "no_advisor_proposals_parsed").sum())

    if diffs.exists():
        count = 0
        for line in diffs.read_text(encoding="utf-8").splitlines():
            if "llm_advised" in line or "advisor_proposal_id" in line or "advisor_batch_id" in line:
                count += 1
        report["advisor_backed_diff_count"] = count

    display(pd.DataFrame([report]))
    if not advisor_csv.empty:
        display(advisor_csv)
    return report

def run_eval_pipeline(mode="smoke2", run_cloud=True, label=None, check=False):
    label = label or f"eval_{mode}_{ts()}"
    out_log = NOTEBOOK_RUN_ROOT / f"{label}.log"
    env = ENV.copy()
    env["RUN_CLOUD"] = "1" if run_cloud else "0"
    cmd = [str(EVAL_PIPELINE), mode, str(BASE_DIR), DEVICE]
    return run_cmd(cmd, env=env, log_path=out_log, check=check)

def locate_latest_eval_pipeline_artifacts():
    roots = sorted((BASE_DIR / "artifacts").glob("eval_pipeline_checks_*"), key=lambda p: p.stat().st_mtime)
    if not roots:
        return None
    root = roots[-1]
    return {
        "root": root,
        "strict_dir": root / "strict_det",
        "cloud_dir": root / "cloud_judge",
        "merge_dir": root / "merged_feedback",
        "advisor_rich_feedback": root / "merged_feedback" / "advisor_rich_feedback.json",
        "summary": root / "check_summary.tsv",
    }

## Cell 1. Local worker and OpenAI preflight

In [ ]:
print("LOCAL_MODEL_DIR:", LOCAL_MODEL_DIR, LOCAL_MODEL_DIR.exists())
assert LOCAL_MODEL_DIR.exists(), LOCAL_MODEL_DIR

if not OPENAI_KEY:
    print("[WARN] No OpenAI key configured. Eval pipeline cloud judge or advisor may fail.")

RUN_WORKER_PREFLIGHT = True
if RUN_WORKER_PREFLIGHT:
    rc, out = run_worker_preflight(check=False)
    print("preflight rc:", rc)
    if "Repo id must be in the form" in out or "worker_crash" in out:
        raise RuntimeError("Worker preflight indicates bad model path or worker crash.")

## Cell 2. Generate strict+cloud merged feedback with eval pipeline

In [ ]:
RUN_MERGED_PREFLIGHT = False  # True로 변경하면 smoke2 eval pipeline 실행

if RUN_MERGED_PREFLIGHT:
    rc, out = run_eval_pipeline(mode="smoke2", run_cloud=True, label=f"merged_eval_smoke2_{ts()}", check=False)
    print("eval rc:", rc)

eval_artifacts = locate_latest_eval_pipeline_artifacts()
eval_artifacts

## Cell 3. Inspect advisor_rich_feedback.json

In [ ]:
def inspect_advisor_rich_feedback(path):
    path = Path(path)
    if not path.exists():
        print("missing:", path)
        return {}
    data = json.loads(path.read_text(encoding="utf-8"))
    print("path:", path)
    if isinstance(data, dict):
        print("top-level keys:", sorted(data.keys()))
        for key in ["metadata", "root_cause_summary", "generation_failure_summary", "evidence_quality_summary", "rows"]:
            val = data.get(key)
            if isinstance(val, list):
                print(key, "len=", len(val))
            elif isinstance(val, dict):
                print(key, "keys=", sorted(val.keys())[:40])
            else:
                print(key, type(val).__name__, str(val)[:300])
    return data

if eval_artifacts and eval_artifacts["advisor_rich_feedback"].exists():
    arf = inspect_advisor_rich_feedback(eval_artifacts["advisor_rich_feedback"])
else:
    print("No advisor_rich_feedback.json found yet. Set RUN_MERGED_PREFLIGHT=True and run previous cell.")

## Cell 4. Baseline row 1 merged-context GA/advisor run

In [ ]:
ROW_NO = 1

MERGED_ADVISOR_ARGS = [
    "--llm-mutation-advisor",
    "--advisor-model-key", "gpt41_mini",
    "--advisor-llm-mode", "openai",
    "--advisor-trigger-mode", "always",
    "--advisor-min-population-for-child", "4",
    "--advisor-force-child-quota",
    "--advisor-compression-child-quota", "1",
    "--advisor-prefer-compression-after-detpass", "90",
    "--advisor-temperature", "0.0",
]

ROW_TUNING = [
    "--population", "4",
    "--gens", "2",
    "--min-generations", "2",
    "--max-generations", "2",
    "--sample-size", "1",
    "--validation-size", "1",
    "--cheap-eval-limit", "1",
    "--plateau-window", "1",
    "--disruptive-max-attempts", "1",
    "--timeout-sec", "2400",
]

merged_row1_baseline = run_ga(
    label=f"merged_row{ROW_NO:03d}_g2_{ts()}",
    scope_args=["--start-row", str(ROW_NO), "--end-row", str(ROW_NO)],
    tuning_args=ROW_TUNING,
    extra_args=MERGED_ADVISOR_ARGS,
)

summarize_ga_run(merged_row1_baseline)
inspect_failures(merged_row1_baseline, max_rows=120)
if detect_worker_crash(merged_row1_baseline):
    raise RuntimeError("worker_crash detected.")

## Cell 5. Feedback / mutation / advisor evidence inspection

In [ ]:
show_prompt_mutation_evidence(merged_row1_baseline)
advisor_effectiveness_report(merged_row1_baseline)

if eval_artifacts and eval_artifacts["summary"].exists():
    print("\nEval pipeline summary:")
    print(eval_artifacts["summary"].read_text(encoding="utf-8")[:5000])

## Cell 6. Actual prompt and advisor prompt inspection

In [ ]:
_ = inspect_prompt_log_from_candidates(merged_row1_baseline, row_index=0, max_chars=9000)

print("\nCloud advisor prompt artifacts:")
for p in sorted(Path(merged_row1_baseline).glob("cloud_advisor_prompt_generation_*.md")):
    print("\n==", p.name, "==")
    print(p.read_text(encoding="utf-8")[:6000])

## Cell 7. Manual merged-feedback patch

In [ ]:
MANUAL_MERGED_RULES = [
    "When strict DET and cloud judge disagree, prefer schema-valid canonical JOILang code over semantic paraphrase.",
    "Use root-cause evidence to patch only the smallest relevant prompt block.",
    "For direct action failures, prioritize canonical service mapping and minimal output schema compliance.",
]

patched_merged_genome = make_manual_patched_genome(
    merged_row1_baseline / "best_genome.json",
    rules=MANUAL_MERGED_RULES,
    target_block_id="02",
    patch_name=f"merged_row{ROW_NO:03d}_manual_rules_{ts()}",
    force_source_file_patch=True,
)

print("patched_merged_genome:", patched_merged_genome)

## Cell 8. Before/after diff

In [ ]:
_ = diff_json_objects(
    merged_row1_baseline / "best_genome.json",
    patched_merged_genome,
    max_lines=260,
)

## Cell 9. Re-test same row

In [ ]:
merged_row1_rerun = run_ga(
    label=f"merged_row{ROW_NO:03d}_manual_patch_g2_{ts()}",
    scope_args=["--start-row", str(ROW_NO), "--end-row", str(ROW_NO)],
    tuning_args=ROW_TUNING,
    extra_args=MERGED_ADVISOR_ARGS,
    genome_json=patched_merged_genome,
)

summarize_ga_run(merged_row1_rerun)
inspect_failures(merged_row1_rerun, max_rows=120)

## Cell 10. Before/after evaluation and prompt comparison

In [ ]:
compare_two_runs(merged_row1_baseline, merged_row1_rerun)
show_prompt_mutation_evidence(merged_row1_rerun)
advisor_effectiveness_report(merged_row1_rerun)
_ = inspect_prompt_log_from_candidates(merged_row1_rerun, row_index=0, max_chars=9000)

## Cell 11. Category sweep

In [ ]:
RUN_MERGED_CATEGORY_SWEEP = False

CATEGORY_TUNING = [
    "--population", "6",
    "--gens", "3",
    "--min-generations", "2",
    "--max-generations", "3",
    "--sample-size", "4",
    "--validation-size", "4",
    "--cheap-eval-limit", "2",
    "--plateau-window", "1",
    "--disruptive-max-attempts", "1",
    "--timeout-sec", "3600",
    "--limit-per-category", "5",
]

merged_category_runs = []
if RUN_MERGED_CATEGORY_SWEEP:
    for cat in range(1, 9):
        rd = run_ga(
            label=f"merged_category{cat}_g3_{ts()}",
            scope_args=["--category", str(cat)],
            tuning_args=CATEGORY_TUNING,
            extra_args=MERGED_ADVISOR_ARGS,
        )
        merged_category_runs.append(rd)
        summarize_ga_run(rd)
        advisor_effectiveness_report(rd)

## Cell 12. Full eval pipeline + full 280 rows × 10 generations

In [ ]:
RUN_MERGED_FULL_EVAL_AND_GA = False

FULL_TUNING = [
    "--population", "16",
    "--gens", "10",
    "--min-generations", "5",
    "--max-generations", "10",
    "--sample-size", "40",
    "--validation-size", "40",
    "--cheap-eval-limit", "20",
    "--plateau-window", "3",
    "--disruptive-max-attempts", "3",
    "--timeout-sec", "7200",
]

merged_full_run = None
if RUN_MERGED_FULL_EVAL_AND_GA:
    rc, out = run_eval_pipeline(mode="full", run_cloud=True, label=f"merged_eval_full_{ts()}", check=False)
    print("full eval rc:", rc)

    eval_artifacts = locate_latest_eval_pipeline_artifacts()
    print("eval artifacts:", eval_artifacts)

    merged_full_run = run_ga(
        label=f"merged_full280_g10_{ts()}",
        scope_args=[
            "--category", "1", "--category", "2", "--category", "3", "--category", "4",
            "--category", "5", "--category", "6", "--category", "7", "--category", "8",
        ],
        tuning_args=FULL_TUNING,
        extra_args=MERGED_ADVISOR_ARGS,
    )
    summarize_ga_run(merged_full_run)
    advisor_effectiveness_report(merged_full_run)

## Cell 13. Final plots

In [ ]:
analysis_runs = []
for name in ["merged_row1_baseline", "merged_row1_rerun", "merged_full_run"]:
    value = globals().get(name)
    if value:
        analysis_runs.append(Path(value))
analysis_runs += merged_category_runs if "merged_category_runs" in globals() else []

display(collect_run_table(analysis_runs))
_ = plot_generation_history(analysis_runs)